In [39]:
# mypy: ignore-errors
# ruff: noqa
import gc
import math
import os
import sys
import warnings

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

warnings.simplefilter("ignore")
from sklearn.model_selection import KFold

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [40]:
# ruff: noqa
%load_ext autoreload
%autoreload 2
from get_params import get_params
from metrics import compute_metrics_stats

from drGAT import drGAT
from drGAT.load_data import load_data
from drGAT.sampler import NewSampler

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [41]:
name = "gdsc1"
# task = "cell"
task = "drug"


method = "Transformer"
PATH = f"../{name}_data/"
target_dim = [
    1,  # Drug
    # 0  # Cell
]

In [42]:
(
    res,
    pos_num,
    null_mask,
    S_d,
    S_c,
    S_g,
    A_cg,
    A_dg,
    _,
    _,
    _,
) = load_data(name)
res = res.T
cell_sum = np.sum(res.values, axis=1)
drug_sum = np.sum(res.values, axis=0)

load gdsc1
unique drugs: 73
unique genes: 166
DTI unique genes:  166
Top 90% variable genes:  1957
Total:  2099
Final gene exp shape: (916, 2099)
Final drug Act shape: (331, 916)


100%|██████████| 25/25 [00:01<00:00, 15.28it/s]


Done!


In [43]:
def drGAT_new(
    res_mat,
    null_mask,
    target_dim,
    target_index,
    S_d,
    S_c,
    S_g,
    A_cg,
    A_dg,
    PATH,
    params,
    device,
    seed,
):
    sampler = NewSampler(
        res_mat,
        null_mask,
        target_dim,
        target_index,
        S_d,
        S_c,
        S_g,
        A_cg,
        A_dg,
        PATH,
        seed,
    )

    val_labels = sampler.test_labels["Label"]

    if len(np.unique(val_labels)) < 2:
        print(f"Target {target_index} skipped: Validation set has only one class.")
        return None, None

    (_, _, _, best_val_labels, best_val_prob, best_metrics, _, _, _) = drGAT.train(
        sampler, params=params, device=device, verbose=False
    )

    return best_val_labels, best_val_prob

In [44]:
PATH = f"../{name}_data/"
params = config = {
    "dropout1": 0.5,
    "dropout2": 0.35,
    "hidden1": 512,
    "hidden2": 248,
    "hidden3": 210,
    "heads": 2,
    "activation": "relu",
    "optimizer": "Adam",
    "lr": 0.0002418546386305523,
    "weight_decay": 7.521978942286918e-06,
    "scheduler": "Cosine",
    "T_max": 29,
    "gnn_layer": method,
}

params.update(
    {
        "n_drug": S_d.shape[0],
        "n_cell": S_c.shape[0],
        "n_gene": S_g.shape[0],
        "epochs": 1,
        "heads": 1,
        "hidden1": 128,
        "hidden2": 128,
        "hidden3": 128,
    }
)

In [45]:
n_kfold = 1
true_datas = pd.DataFrame()
predict_datas = pd.DataFrame()
for dim in target_dim:
    for seed, target_index in tqdm(enumerate(np.arange(res.shape[dim]))):
        if dim:
            if drug_sum[target_index] < 10:
                continue
        else:
            if cell_sum[target_index] < 10:
                continue

        true_data, predict_data = drGAT_new(
            res_mat=res,
            null_mask=null_mask.T.values,
            target_dim=dim,
            target_index=target_index,
            S_d=S_d,
            S_c=S_c,
            S_g=S_g,
            A_cg=A_cg,
            A_dg=A_dg,
            PATH=PATH,
            params=params,
            device=device,
            seed=seed,
        )
        true_datas = pd.concat(
            [true_datas, pd.DataFrame(true_data).T], ignore_index=True
        )
        predict_datas = pd.concat(
            [predict_datas, pd.DataFrame(predict_data).T], ignore_index=True
        )

0it [00:00, ?it/s]

Using device: cuda


1it [00:04,  4.51s/it]

Best model found at epoch 1
Using device: cuda


2it [00:08,  4.48s/it]

...

Best model found at epoch 1
Using device: cuda


327it [15:28,  3.04s/it]

Best model found at epoch 1
Using device: cuda


331it [15:32,  2.82s/it]

Best model found at epoch 1


In [50]:
sampler = NewSampler(
    res,
    null_mask.T.values,
    target_dim,
    target_index,
    S_d,
    S_c,
    S_g,
    A_cg,
    A_dg,
    PATH,
    seed,
)

In [ ]:
# true_datas.to_csv(f"new_true_drug_{method}_{name}.csv")
# predict_datas.to_csv(f"new_pred_drug_{method}_{name}.csv")

In [47]:
compute_metrics_stats(
    true=true_datas,
    pred=predict_datas,
    target_metrics=["AUROC", "AUPR", "F1", "ACC"],
)

{'raw': {'means': {'ACC': 0.42357924948395537,
   'Precision': 0.07861547721146137,
   'Recall': 0.04662125094451214,
   'F1': 0.03448727106663517,
   'AUROC': 0.5220946088346943,
   'AUPR': 0.6170600010807088,
   'MCC': 0.0006652564199394385,
   'Specificity': 0.9538518397040746,
   'Balanced_ACC': 0.5002365453242933,
   'LogLoss': 0.9832482796958497,
   'Brier': 0.36406527317015763},
  'stds': {'ACC': 0.15756580744448234,
   'Precision': 0.23598203099942808,
   'Recall': 0.20679064073004388,
   'F1': 0.14429198636443005,
   'AUROC': 0.10125448786422153,
   'AUPR': 0.16054296945903132,
   'MCC': 0.02256019026139098,
   'Specificity': 0.2064633812369658,
   'Balanced_ACC': 0.004136990635104383,
   'LogLoss': 0.2907580648745487,
   'Brier': 0.10478478156366147}},
 'formatted': {'ACC': '0.424 (±0.158)',
  'Precision': '0.079 (±0.236)',
  'Recall': '0.047 (±0.207)',
  'F1': '0.034 (±0.144)',
  'AUROC': '0.522 (±0.101)',
  'AUPR': '0.617 (±0.161)',
  'MCC': '0.001 (±0.023)',
  'Specificity

In [21]:
import pandas as pd
from sklearn.metrics import (accuracy_score, average_precision_score,
                             balanced_accuracy_score, brier_score_loss,
                             cohen_kappa_score, f1_score, fbeta_score,
                             log_loss, matthews_corrcoef, precision_score,
                             recall_score, roc_auc_score)

In [24]:
res = pd.DataFrame()
for i in range(true_datas.shape[0]):
    # データ前処理
    true_labels = true_datas.loc[i].dropna()
    pred_values = predict_datas.loc[i].dropna()
    pred_labels = np.round(pred_values).astype(int)

    # メトリクス計算
    metrics = {
        "ACC": accuracy_score(true_labels, pred_labels),
        "Precision": precision_score(true_labels, pred_labels, zero_division=0),
        "Recall": recall_score(true_labels, pred_labels, zero_division=0),
        "F1": f1_score(true_labels, pred_labels, zero_division=0),
        "AUROC": roc_auc_score(true_labels, pred_values),
        "AUPR": average_precision_score(true_labels, pred_values),
        "MCC": matthews_corrcoef(true_labels, pred_labels),
        "Specificity": recall_score(
            true_labels, pred_labels, pos_label=0, zero_division=0
        ),
        "Balanced_ACC": balanced_accuracy_score(true_labels, pred_labels),
        "LogLoss": log_loss(true_labels, pred_values),
        "Brier": brier_score_loss(true_labels, pred_values),
    }
    res = pd.concat([res, pd.DataFrame([metrics])], ignore_index=True)

In [25]:
res

,ACC,Precision,Recall,F1,AUROC,AUPR,MCC,Specificity,Balanced_ACC,LogLoss,Brier
0,0.484375,0.488152,0.643750,0.555256,0.478340,0.508872,-0.032970,0.325000,0.484375,0.700010,0.253373
1,0.500000,0.500000,1.000000,0.666667,0.507183,0.508660,0.000000,0.000000,0.500000,0.703527,0.255110
2,0.500000,0.000000,0.000000,0.000000,0.642857,0.580294,0.000000,1.000000,0.500000,0.753349,0.278364
3,0.500000,0.000000,0.000000,0.000000,0.501730,0.524149,0.000000,1.000000,0.500000,0.701630,0.254218
4,0.522727,0.512821,0.909091,0.655738,0.442149,0.449333,0.071611,0.136364,0.522727,0.714565,0.260107
...,...,...,...,...,...,...,...,...,...,...,...
124,0.500000,0.000000,0.000000,0.000000,0.506958,0.495335,0.000000,1.000000,0.500000,0.786951,0.292423
125,0.500000,0.000000,0.000000,0.000000,0.616690,0.567900,0.000000,1.000000,0.500000,0.716329,0.261432
126,0.500000,0.000000,0.000000,0.000000,0.499408,0.506662,0.000000,1.000000,0.500000,0.709854,0.258217
127,0.500000,0.500000,0.842105,0.627451,0.434287,0.456675,0.000000,0.157895,0.500000,0.721967,0.264075
